# 04 - Unified evaluation (Phase A)



## Scoring rule 

- **Uncertain truth (-1) is excluded** - U-Ignore, following Irvin et al. (2019).
- **Blank truth is treated as absent.** binarisation choice, not part of U-Ignore.
- **Prediction: only "present" counts as positive.** absent, uncertain, not_mentioned, and failed output all count as absent.
- **All reports are scored.** Failed output is counted as absent and never dropped.
- **Primary metric: macro F1 over the 13 clinical labels.** No Finding is reported separately; macro14 is shown only for continuity with notebooks 02/03.
- **Uncertainty: paired bootstrap over reports**, 2,000 resamples, identical resampled indices across runs.
- **Primary few-shot comparison uses seeds 42/123/456** (all Colab). Seeds 789/1011 ran structure on Kaggle and direct on Colab; they are sensitivity/replication only.
- **Limitation:** direct CSVs contain no raw output, so their usable % is an all-NaN proxy. Final runs must save raw output for both pipelines.



In [1]:

from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/cxr_eval_data"

NAMES = [
 "structure_zero_shot_results.csv",
 "structure_few_shot_seed42_results.csv",
 "structure_few_shot_seed123_results.csv",
 "structure_few_shot_seed456_results.csv",
 "structure_few_shot_seed789_results.csv",
 "structure_few_shot_seed1011_results.csv",
 "direct_zero_shot_results.csv",
 "direct_few_shot_seed42_results.csv",
 "direct_few_shot_seed123_results.csv",
 "direct_few_shot_seed456_results.csv",
 "direct_few_shot_seed789_results.csv",
 "direct_few_shot_seed1011_results.csv",
]

def find(name):
    for d in (BASE, os.path.join(BASE, "results")):
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return None

missing = [n for n in NAMES if find(n) is None]
print(f"Found {len(NAMES) - len(missing)}/{len(NAMES)} files in {BASE}")
if missing:
    print("Missing:")
    for n in missing:
        print("  " + n)
    raise FileNotFoundError("Upload the missing files into the Drive folder, then rerun this cell.")


Mounted at /content/drive
Found 12/12 files in /content/drive/MyDrive/cxr_eval_data


In [ ]:
# Unified scorer 
import os, json
import pandas as pd, numpy as np

PC = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion',
      'Lung Opacity','Edema','Consolidation','Pneumonia','Atelectasis',
      'Pneumothorax','Pleural Effusion','Pleural Other','Fracture','Support Devices']
CLIN = [p for p in PC if p != 'No Finding']

PATH_COLS = {
 "lungs_and_airways.lung_opacity":"Lung Opacity","lungs_and_airways.consolidation":"Consolidation",
 "lungs_and_airways.edema":"Edema","lungs_and_airways.atelectasis":"Atelectasis",
 "lungs_and_airways.lung_lesion":"Lung Lesion","lungs_and_airways.pneumonia":"Pneumonia",
 "pleura.pleural_effusion":"Pleural Effusion","pleura.pneumothorax":"Pneumothorax",
 "pleura.pleural_other":"Pleural Other","cardiovascular.cardiomegaly":"Cardiomegaly",
 "cardiovascular.enlarged_cardiomediastinum":"Enlarged Cardiomediastinum",
 "bones_and_chest_wall.fracture":"Fracture","devices.support_devices":"Support Devices",
 "general.no_finding":"No Finding"}
VMAP = {"present":"1","absent":"0","uncertain":"-1","not_mentioned":"NaN","nan":"NaN"}

def parse_struct(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e <= s: return ["NaN"]*14
    try: d = json.loads(raw[s:e+1])
    except Exception: return ["NaN"]*14
    out = {}
    for path, p in PATH_COLS.items():
        cur = d
        for k in path.split("."):
            cur = cur.get(k) if isinstance(cur, dict) else None
        out[p] = VMAP.get(cur.strip().lower(), "NaN") if isinstance(cur, str) else "NaN"
    return [out.get(p, "NaN") for p in PC]

def load(path, kind):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    pred = [parse_struct(r) for r in df["raw"]] if kind == "S" else \
           [[df[f"pred_{p}"].iloc[i] for p in PC] for i in range(len(df))]
    truth = {}
    for p in PC:
        t = df[f"truth_{p}"].map(lambda x: np.nan if x in ("", "nan") else float(x)).to_numpy()
        truth[p] = np.where(t == -1, np.nan, np.where(np.isnan(t), 0.0, t))
    pos = {p: np.array([1.0 if row[PC.index(p)] in ("1", "1.0") else 0.0 for row in pred]) for p in PC}
    usable = float(np.mean([any(v not in ("NaN", "nan") for v in row) for row in pred]))
    return {"df": df, "truth": truth, "pos": pos, "usable": usable}

def f1(t, pr):
    m = ~np.isnan(t)
    tp = ((t == 1) & (pr == 1) & m).sum()
    fp = ((t == 0) & (pr == 1) & m).sum()
    fn = ((t == 1) & (pr == 0) & m).sum()
    d = 2*tp + fp + fn
    return 2*tp/d if d else 0.0

def evaluate(r):
    per = {p: f1(r["truth"][p], r["pos"][p]) for p in PC}
    return {"usable": r["usable"], "macro13": np.mean([per[p] for p in CLIN]),
            "macro14": np.mean(list(per.values())), "No Finding": per["No Finding"], **per}

FILES = {
 "S zero-shot": (find("structure_zero_shot_results.csv"), "S"),
 "S seed42":   (find("structure_few_shot_seed42_results.csv"), "S"),
 "S seed123":  (find("structure_few_shot_seed123_results.csv"), "S"),
 "S seed456":  (find("structure_few_shot_seed456_results.csv"), "S"),
 "S seed789":  (find("structure_few_shot_seed789_results.csv"), "S"),
 "S seed1011": (find("structure_few_shot_seed1011_results.csv"), "S"),
 "D zero-shot": (find("direct_zero_shot_results.csv"), "D"),
 "D seed42":   (find("direct_few_shot_seed42_results.csv"), "D"),
 "D seed123":  (find("direct_few_shot_seed123_results.csv"), "D"),
 "D seed456":  (find("direct_few_shot_seed456_results.csv"), "D"),
 "D seed789":  (find("direct_few_shot_seed789_results.csv"), "D"),
 "D seed1011": (find("direct_few_shot_seed1011_results.csv"), "D"),
}

runs = {k: load(*v) for k, v in FILES.items()}
ids = runs["S zero-shot"]["df"]["id"].tolist()
for k, r in runs.items():
    assert r["df"]["id"].tolist() == ids, k
    for p in PC:
        assert np.allclose(r["truth"][p], runs["S zero-shot"]["truth"][p], equal_nan=True), (k, p)

res = {k: evaluate(r) for k, r in runs.items()}
print(f"{'run':12s} {'usable%':>8s} {'macro13':>8s} {'macro14':>8s} {'NoFind':>7s}")
print("-" * 48)
for k, m in res.items():
    print(f"{k:12s} {100*m['usable']:>6.1f}% {m['macro13']:>8.3f} {m['macro14']:>8.3f} {m['No Finding']:>7.3f}")
pd.DataFrame(res).T.to_csv(os.path.join(BASE, "unified_metrics.csv"))
print(f"\nSaved {os.path.join(BASE, 'unified_metrics.csv')}")

def boot(groups, label, B=2000, seed=42):
    n = len(ids); rng = np.random.default_rng(seed)
    idxs = [rng.choice(n, n, replace=True) for _ in range(B)]
    def macro(r, idx):
        vals = []
        for p in CLIN:
            t, pr = r["truth"][p][idx], r["pos"][p][idx]; m = ~np.isnan(t)
            tp = ((t == 1) & (pr == 1) & m).sum(); fp = ((t == 0) & (pr == 1) & m).sum()
            fn = ((t == 1) & (pr == 0) & m).sum()
            d = 2*tp + fp + fn
            vals.append(2*tp/d if d else 0.0)
        return np.mean(vals)
    diffs = np.array([np.mean([macro(runs[a], idx) - macro(runs[b], idx) for a, b in groups]) for idx in idxs])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"  {label}: {diffs.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]  P(>0)={np.mean(diffs > 0):.2f}")

print("\nPaired bootstrap, macro13 (2000 resamples, shared report indices):")
boot([("S zero-shot", "D zero-shot")], "zero-shot, structure - direct")
print("Few-shot, same platform only (seeds 42/123/456) - primary:")
boot([("S seed42", "D seed42"), ("S seed123", "D seed123"), ("S seed456", "D seed456")], "mean over seeds 42/123/456")
print("Few-shot, all five seeds - sensitivity only (platform-confounded):")
boot([("S seed42", "D seed42"), ("S seed123", "D seed123"), ("S seed456", "D seed456"), ("S seed789", "D seed789"), ("S seed1011", "D seed1011")], "mean over all five seeds")


run           usable%  macro13  macro14  NoFind
------------------------------------------------
S zero-shot    99.5%    0.456    0.467   0.615
S seed42       91.6%    0.418    0.441   0.744
S seed123      97.2%    0.422    0.445   0.748
S seed456      94.7%    0.458    0.478   0.740
S seed789      81.0%    0.266    0.297   0.696
S seed1011     67.3%    0.307    0.340   0.764
D zero-shot    44.1%    0.423    0.393   0.000
D seed42      100.0%    0.441    0.452   0.594
D seed123     100.0%    0.462    0.472   0.609
D seed456      99.7%    0.463    0.470   0.565
D seed789     100.0%    0.387    0.399   0.562
D seed1011    100.0%    0.529    0.533   0.577

Saved /content/drive/MyDrive/cxr_eval_data/unified_metrics.csv

Paired bootstrap, macro13 (2000 resamples, shared report indices):
  zero-shot, structure - direct: +0.034 [-0.020, +0.091]  P(>0)=0.89
Few-shot, same platform only (seeds 42/123/456) - primary:
  mean over seeds 42/123/456: -0.024 [-0.070, +0.021]  P(>0)=0.14
Few-shot, all